[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day3_solution.ipynb)

# Day 3 · 정답 — 머신러닝

경사 하강법 · scikit-learn · 검증과 평가 · 회귀

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`lecture` 와 `practice` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 학습의 원리

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 1.** 학습률 `lr` 을 `0.01` 로 두고 20회를 돌린 뒤 `x` 를 확인한다.
값이 3에 **덜 가까워지는 것**을 본다.

In [ ]:
def grad(x): return 2 * (x - 3)
x = 10.0
lr = 0.01
for _ in range(20):
    x = x - lr * grad(x)

assert x > 4, f'학습률이 작으면 20회로는 못 간다. 실제 {x}'
print('통과 — x =', round(x, 3))

> **실습문제 2.** 이번엔 학습률 `lr` 을 `1.1` 로 두고 같은 20회를 돌린다. `x` 가 **발산**한다.

In [ ]:
def grad(x): return 2 * (x - 3)
x = 10.0
lr = 1.1
for _ in range(20):
    x = x - lr * grad(x)

assert abs(x) > 100, f'학습률이 너무 크면 튕겨 나간다. 실제 {x}'
print('통과 — x =', round(x, 1))

## 2. scikit-learn — 네 줄로 끝나는 학습

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 3.** 로지스틱 회귀를 학습하고 **테스트 정확도**를 `acc` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LogisticRegression
X_tr, X_te, y_tr, y_te = split()
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
model = LogisticRegression(max_iter=1000)
model.fit(X_tr_s, y_tr)
acc = model.score(X_te_s, y_te)

assert acc > 0.85, f'0.85 는 넘어야 한다. 실제 {acc}'
print('통과 — 정확도', round(acc, 3))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** 결정 트리를 `max_depth=3` 으로 학습하고 정확도를 `acc` 에 담는다.
> 트리는 스케일링이 필요 없다. 원본 `X_tr` 을 그대로 넣는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.tree import DecisionTreeClassifier
X_tr, X_te, y_tr, y_te = split()
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_tr, y_tr)
acc = model.score(X_te, y_te)

assert acc > 0.88, f'실제 {acc}'
print('통과 — 정확도', round(acc, 3))

> **빈칸 문제 2.** 랜덤 포레스트를 학습하고 **중요도가 가장 높은 열**의 이름을 `top` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.ensemble import RandomForestClassifier
X_tr, X_te, y_tr, y_te = split()
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_tr, y_tr)
pairs = sorted(zip(model.feature_importances_, X_tr.columns), reverse=True)
top = pairs[0][1]

assert top == '소성온도', f'기대 소성온도, 실제 {top}'
print('통과')

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 4.** 세 모델(로지스틱·결정 트리·랜덤 포레스트)의 테스트 정확도를 재어
`scores` 딕셔너리에 담고 출력한다.
> 로지스틱만 스케일링한 데이터를 쓴다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
X_tr, X_te, y_tr, y_te = split()
sc = StandardScaler()
X_tr_s, X_te_s = sc.fit_transform(X_tr), sc.transform(X_te)
scores = {}
scores['logistic'] = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr).score(X_te_s, y_te)
scores['tree'] = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr).score(X_te, y_te)
scores['forest'] = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr).score(X_te, y_te)
for k, v in scores.items():
    print(f'{k:>10} {v:.3f}')
assert set(scores) == {'logistic', 'tree', 'forest'}, f'키 확인: {scores.keys()}'
assert scores['forest'] > scores['logistic'], '포레스트가 로지스틱보다 낫다'
print('통과')

## 3. 검증과 평가

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 5.** **전부 양품이라 찍는** 예측을 만들어 정확도를 `dumb_acc` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
import numpy as np
X_tr, X_te, y_tr, y_te = split()
pred = np.ones(len(y_te), dtype=int)
dumb_acc = (pred == y_te).mean()

assert abs(dumb_acc - 0.81) < 0.02, f'실제 {dumb_acc}'
print('통과 — 아무것도 안 배워도', round(dumb_acc, 3))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 3.** 불량(`0`)을 **놓친 건수**를 `missed` 에 담는다.
> 실제 불량인데 양품이라 예측한 것이다. 혼동 행렬의 어느 칸인지 생각한다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
X_tr, X_te, y_tr, y_te = split()
model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
pred = model.predict(X_te)
cm = confusion_matrix(y_te, pred)
missed = cm[0, 1]

assert missed == ((y_te == 0) & (pred == 1)).sum(), f'실제 놓친 수와 다르다: {missed}'
print('통과 — 놓친 불량', missed, '건')

> **빈칸 문제 4.** 불량을 양성으로 놓고 **재현율**을 구해 `recall` 에 담는다.
> `recall_score(..., pos_label=0)` 이다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score
X_tr, X_te, y_tr, y_te = split()
model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
pred = model.predict(X_te)
recall = recall_score(y_te, pred, pos_label=0)

assert 0.6 < recall < 1.0, f'실제 {recall}'
print('통과 — 재현율', round(recall, 3))

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 6.** **과적합**을 눈으로 본다. 결정 트리의 `max_depth` 를 1부터 20까지 늘리며
훈련 정확도와 테스트 정확도를 같이 재어 그린다.
> 훈련은 계속 오르는데 테스트는 어느 지점부터 안 오른다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt
X_tr, X_te, y_tr, y_te = split()
depths = range(1, 21)
tr, te = [], []
for d_ in depths:
    m = DecisionTreeClassifier(max_depth=d_, random_state=42).fit(X_tr, y_tr)
    tr.append(m.score(X_tr, y_tr))
    te.append(m.score(X_te, y_te))
plt.plot(depths, tr, label='train')
plt.plot(depths, te, label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.legend(); plt.show()
assert tr[-1] > te[-1], '깊어질수록 훈련 점수가 테스트보다 높아진다'
assert tr[-1] > 0.99, f'끝에서 훈련 정확도는 1에 가깝다: {tr[-1]}'
print('통과')

## 4. 회귀 — 용량 맞히기

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 5.** 선형 회귀로 `방전용량` 을 예측하고 **R²** 를 `r2` 에 담는다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
X_tr, X_te, y_tr, y_te = split(target='방전용량')
model = LinearRegression()
model.fit(X_tr, y_tr)
r2 = r2_score(y_te, model.predict(X_te))

assert r2 > 0.65, f'실제 {r2}'
print('통과 — R2', round(r2, 3))

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 7.** 선형 회귀보다 랜덤 포레스트가 더 잘 맞히는 것을 확인한다.
두 R² 를 `r2_lin`, `r2_rf` 에 담고 차이를 출력한다.
> 온도와 용량의 관계가 **곡선**이라 그렇다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
X_tr, X_te, y_tr, y_te = split(target='방전용량')
r2_lin = r2_score(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))
r2_rf = r2_score(y_te, RandomForestRegressor(n_estimators=200, random_state=42)
                       .fit(X_tr, y_tr).predict(X_te))
print(round(r2_lin, 3), round(r2_rf, 3), '차이', round(r2_rf - r2_lin, 3))
assert r2_rf > r2_lin, '포레스트가 더 높아야 한다'
print('통과')

## 5. 종합 문제

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 8.** **놓친 불량을 줄이는** 쪽으로 판정 기준을 옮긴다.
`predict_proba` 로 양품 확률을 얻고, 기준을 0.5 대신 **0.7** 로 올려
불량 재현율이 오르는지 확인한다.
> 확률이 0.7 미만이면 불량으로 본다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, precision_score
X_tr, X_te, y_tr, y_te = split()
model = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
base = (proba >= 0.5).astype(int)
strict = (proba >= 0.7).astype(int)
r0 = recall_score(y_te, base, pos_label=0)
r1 = recall_score(y_te, strict, pos_label=0)
p0 = precision_score(y_te, base, pos_label=0)
p1 = precision_score(y_te, strict, pos_label=0)
print(f'기준 0.5 — 재현율 {r0:.3f} 정밀도 {p0:.3f}')
print(f'기준 0.7 — 재현율 {r1:.3f} 정밀도 {p1:.3f}')
assert r1 >= r0, '기준을 올리면 불량을 더 많이 잡는다'
assert p1 <= p0, '대신 헛경보가 늘어 정밀도는 떨어진다'
print('통과')

> **실습문제 9.** 어떤 열이 없어도 되는지 본다.
`성형압력` 을 **뺀 채로** 랜덤 포레스트를 학습해 정확도가 거의 그대로인 것을 확인한다.
> `성형압력` 는 용량과 상관이 −0.05 였다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def load():
    df = pd.read_csv('https://tunalee.github.io/posco/data/batch_quality.csv', thousands=',', na_values=['N/A', '-'])
    df['설비호기'] = df['설비호기'].str.strip().str.upper()
    df['입도'] = df['입도'].fillna(df['입도'].median())
    df = df.dropna(subset=['수분율'])
    return pd.get_dummies(df, columns=['설비호기', '교대조'], drop_first=True)

def split(target='양품여부'):
    d = load()
    drop = ['양품여부', '방전용량', '배치번호']
    X, y = d.drop(columns=drop), d[target]
    return train_test_split(X, y, test_size=0.2, random_state=42,
                            stratify=y if target == '양품여부' else None)
from sklearn.ensemble import RandomForestClassifier
X_tr, X_te, y_tr, y_te = split()
full = RandomForestClassifier(n_estimators=200, random_state=42)\
        .fit(X_tr, y_tr).score(X_te, y_te)
less = RandomForestClassifier(n_estimators=200, random_state=42)\
        .fit(X_tr.drop(columns=['성형압력']), y_tr)\
        .score(X_te.drop(columns=['성형압력']), y_te)
print(round(full, 3), round(less, 3), '차이', round(full - less, 3))
assert abs(full - less) < 0.03, f'press 를 빼도 크게 안 변한다: {full} vs {less}'
print('통과')